In [1]:
import pandas as pd

### 1.2 Identify Fact and Dimensions

**Fact Table:**
*   **Fact_Transactions**: Contains the measurable metrics and the foreign keys required to link to the dimensions.
    *   *Measure:* `Unit` (quantity of traded shares, extracted from the `account-statement` file).
    *   *Degenerate Dimension:* `IDTransaction` (transaction identifier, from the `account-statement` file).
    *   *Foreign Keys:* `Time_ID`, `Symbol_ID`, `Geography_ID`, `TransactionType_ID`.

**Dimension Tables (Descriptive Attributes):**
*   **Dim_Time**: Derived from the `Date` column of the `account-statement` file.
    *   *Attributes:* 
    * `Time_ID` (Surrogate Key), 
    *   `Date`, 
    *   `Day`, 
    *   `Month`, 
    *   `Quarter`, 
    *   `Year`.

*   **Dim_Symbol**: Built from the `symbols.csv` dataset.
    *   *Attributes:* 
    *   `Symbol_ID` (Surrogate Key), 
    *   `Symbol`, 
    *   `Company_Name`, 
    *   `Sector`, 
    *   `Industry`.
    *   *Modeling Choice:* The `country` attribute originally found in `symbols.csv` was intentionally excluded from this dimension. Instead, it is used during the ETL process to map the data to `Dim_Geography`. This guarantees no attribute duplication between dimensions.

*   **Dim_Geography**: Built from the `country.csv` metadata dataset.
    *   *Attributes:* 
    *   `Geography_ID` (Surrogate Key), 
    *   `Country_Name` (from the `name` column), 
    *   `Region`, 
    *   `Sub_region`.

*   **Dim_TransactionType**: Extracted from the `TransactionType` column of the `account-statement` file.
    *   *Attributes:* 
    *   `TransactionType_ID` (Surrogate Key), 
    *   `TransactionType` (values: BUY or SELL).

### 1.3 Define Dimension Hierarchies
The data within the dimension tables have been structured into the following logical hierarchies. 

*   **Dim_Time hierarchy**: Day -> Month -> Quarter -> Year
*   **Dim_Geography hierarchy**: Country -> Sub-region -> Region 
*   **Dim_Symbol hierarchy**: Sector -> Industry -> Symbol (Company)
    *(Custom hierarchy defined based on the attributes available in the symbols.csv dataset. This allows for broad analysis at the macroeconomic sector level, drilling down to specific industries, and finally to individual companies).*
*   **Dim_TransactionType**: Flat dimension. 
    *(No hierarchy is applicable here, as it strictly represents the discrete "BUY" or "SELL" nature of the transaction).*

### 1.4 Design the Star Schema
The star schema is designed with the `Fact_Transactions` table at the center with the four dimension tables. 

**1. Surrogate Keys for Dimensions:**
To ensure robust data modeling and avoid relying on natural/business keys, an auto-incrementing integer Surrogate Key (SK) is defined as the Primary Key (PK) for each dimension table:
*   `Time_ID` (PK for `Dim_Time`)
*   `Geography_ID` (PK for `Dim_Geography`)
*   `Symbol_ID` (PK for `Dim_Symbol`)
*   `TransactionType_ID` (PK for `Dim_TransactionType`)

**2. Mapping Fact Table Foreign Keys:**
The central fact table (`Fact_Transactions`) acts as the junction point. The surrogate keys generated above are mapped inside it as Foreign Keys (FKs) to establish 1-to-Many relationships:
*   `Fact_Transactions.Time_ID` -> references `Dim_Time.Time_ID`
*   `Fact_Transactions.Geography_ID` -> references `Dim_Geography.Geography_ID`
*   `Fact_Transactions.Symbol_ID` -> references `Dim_Symbol.Symbol_ID`
*   `Fact_Transactions.TransactionType_ID` -> references `Dim_TransactionType.TransactionType_ID`

**3. Measures and Descriptive Attributes:**
The schema strictly separates quantitative data from qualitative descriptions:
*   **Measures:** The fact table contains the only numeric, aggregatable metric: `Unit` (the volume of shares traded) [2]. It also holds `IDTransaction` [2] as a *degenerate dimension* to preserve the original transaction identifier without needing a separate dimension table.
*   **Descriptive Attributes:** All textual, filtering, and grouping information is isolated within the dimension tables. This includes temporal details (`Month`, `Quarter`), geographic hierarchies (`Region`, `Country`), company metadata (`Sector`, `Industry`), and the transaction nature (`BUY`/`SELL`).

In [ ]:
# STEP 0: LOAD RAW DATASETS
# Loading CSV files into raw dataframes
df_transactions = pd.read_csv("data/account-statement-1-1-2024-12-31-2024.csv", sep=";")
df_symbols = pd.read_csv("data/symbols.csv", sep=";")
df_country = pd.read_csv("data/country.csv", sep=",")

if 'Unnamed: 5' in df_transactions.columns:
    df_transactions = df_transactions.drop(columns=['Unnamed: 5'])


In [3]:
# A) Check Missing Values 
print("MISSING VALUES ")
print("Transactions:\n", df_transactions.isnull().sum())
print("\nSymbols:\n", df_symbols.isnull().sum())
print("\nCountries:\n", df_country[['name', 'sub-region', 'region']].isnull().sum())

# B) Verify Symbol mapping
missing_symbols = set(df_transactions['Symbol']) - set(df_symbols['symbol'])

print(f"Symbols in transactions NOT present in symbols.csv: {len(missing_symbols)}")
if len(missing_symbols) > 0:
    print("Missing symbols:", missing_symbols)

# C) Verify Country mapping 

missing_countries = set(df_symbols['country']) - set(df_country['name'])

print(f"Countries in symbols.csv NOT present in country.csv: {len(missing_countries)}")
if len(missing_countries) > 0:
    print("Missing countries (Data Quality Issue):", missing_countries)


MISSING VALUES 
Transactions:
 IDTransaction      464
Date               464
TransactionType    464
Symbol             464
Unit               464
dtype: int64

Symbols:
 symbol          0
company_name    0
sector          0
industry        0
country         0
dtype: int64

Countries:
 name          0
sub-region    2
region        2
dtype: int64
Symbols in transactions NOT present in symbols.csv: 19
Missing symbols: {'CSIQ', 'AZM', 'UCG', 'MFG', 'SAP', nan, 'ARCH', 'RIGZU', 'CCAP', 'AGO.l', 'RCMT', 'HTGC', 'VWS', 'FNC', 'MONC', 'WF', 'TKC', 'IBE', 'OBDC'}
Countries in symbols.csv NOT present in country.csv: 2
Missing countries (Data Quality Issue): {'Taiwan', 'Turkey'}


In [4]:

# 0. FIXING DATA ISSUES (Country Names)
# Align country names in symbols.csv with the names in country.csv
country_mapping = {
    "United States": "United States of America",
    "United Kingdom": "United Kingdom of Great Britain and Northern Ireland",
    "South Korea": "Korea, Republic of",
    "Russia": "Russian Federation",
    "Taiwan": "Taiwan, Province of China",
    "Macau": "Macao",
    "Ivory Coast": "Côte d'Ivoire"
}
df_symbols['country'] = df_symbols['country'].replace(country_mapping)


# 1. CREATE DIMENSION TIME

df_transactions['Date'] = pd.to_datetime(df_transactions['Date'], format='%d/%m/%Y %H:%M:%S')

# Extract unique dates and build the required hierarchy
Dim_Time = pd.DataFrame({'Date': df_transactions['Date'].dt.date.unique()})
Dim_Time['Day'] = pd.to_datetime(Dim_Time['Date']).dt.day
Dim_Time['Month'] = pd.to_datetime(Dim_Time['Date']).dt.month
Dim_Time['Quarter'] = pd.to_datetime(Dim_Time['Date']).dt.quarter
Dim_Time['Year'] = pd.to_datetime(Dim_Time['Date']).dt.year

# Generate Surrogate Key (Time_ID)
Dim_Time.insert(0, 'Time_ID', range(1, 1 + len(Dim_Time)))


# 2. CREATE DIMENSION GEOGRAPHY

# Keep only the required columns and drop duplicates
Dim_Geography = df_country[['name', 'region', 'sub-region']].copy()
Dim_Geography.rename(columns={'name': 'Country_Name', 'sub-region': 'Sub_region', 'region': 'Region'}, inplace=True)
Dim_Geography = Dim_Geography.drop_duplicates(subset=['Country_Name']).reset_index(drop=True)

# Generate Surrogate Key (Geography_ID)
Dim_Geography.insert(0, 'Geography_ID', range(1, 1 + len(Dim_Geography)))


# 3. CREATE DIMENSION TRANSACTIONTYPE

Dim_TransactionType = pd.DataFrame({'TransactionType': df_transactions['TransactionType'].unique()})
# Generate Surrogate Key (TransactionType_ID)
Dim_TransactionType.insert(0, 'TransactionType_ID', range(1, 1 + len(Dim_TransactionType)))

print("Dimension Time, Dimension Geography and Dimension TransactionType created")


# 4. CREATE DIMENSION SYMBOL

# Temporarily merge symbols with geography to retrieve the Geography_ID
Dim_Symbol_temp = df_symbols.merge(Dim_Geography, left_on='country', right_on='Country_Name', how='left')

# Isolate only the dimension columns (excluding country to avoid duplication)
Dim_Symbol = Dim_Symbol_temp[['symbol', 'company_name', 'sector', 'industry']].copy()
Dim_Symbol.rename(columns={'symbol': 'Symbol', 'company_name': 'Company_Name', 'sector': 'Sector', 'industry': 'Industry'}, inplace=True)

# Generate Surrogate Key (Symbol_ID)
Dim_Symbol.insert(0, 'Symbol_ID', range(1, 1 + len(Dim_Symbol)))

Map_Symbol_Geo = pd.concat([Dim_Symbol['Symbol_ID'], Dim_Symbol_temp[['symbol', 'Geography_ID']]], axis=1)


# 5. CREATE THE TABLE FACT TRANSACTIONS

Fact_Transactions = df_transactions.copy()

Fact_Transactions['Date_Only'] = Fact_Transactions['Date'].dt.date

Fact_Transactions = Fact_Transactions.merge(Dim_Time[['Time_ID', 'Date']], left_on='Date_Only', right_on='Date', how='left')

Fact_Transactions = Fact_Transactions.merge(Dim_TransactionType, on='TransactionType', how='left')

Fact_Transactions = Fact_Transactions.merge(Map_Symbol_Geo, left_on='Symbol', right_on='symbol', how='left')

# Clean the Fact Table, keeping only the Measure and Foreign Keys
Fact_Transactions = Fact_Transactions[['IDTransaction', 'Time_ID', 'Symbol_ID', 'Geography_ID', 'TransactionType_ID', 'Unit']]

print("Dimension Symbol and Fact_Transactions created")
print("\nFirst 15 rows of the Star Schema Fact Table:")
display(Fact_Transactions.head(15))

Dimension Time, Dimension Geography and Dimension TransactionType created
Dimension Symbol and Fact_Transactions created

First 15 rows of the Star Schema Fact Table:


,IDTransaction,Time_ID,Symbol_ID,Geography_ID,TransactionType_ID,Unit
0,2.769834e+09,1,284.0,175.0,1,1605.0
1,2.767325e+09,2,284.0,175.0,2,1605.0
2,2.815474e+09,3,284.0,175.0,2,914.0
3,2.622244e+09,4,4.0,25.0,1,646.0
4,2.629871e+09,4,258.0,131.0,2,646.0
5,2.782801e+09,5,4.0,25.0,1,519.0
6,2.625828e+09,6,258.0,131.0,2,519.0
7,2.608926e+09,6,4.0,25.0,1,515.0
8,2.633048e+09,7,258.0,131.0,2,515.0
9,2.653834e+09,1,4.0,25.0,1,320.0


In [5]:
# PREPARATION: DENORMALIZE THE STAR SCHEMA FOR ANALYSIS

# Merge the Fact table with all dimensions to create a dataframe 
df_analysis = Fact_Transactions.merge(Dim_Time, on='Time_ID', how='inner') \
                               .merge(Dim_Symbol, on='Symbol_ID', how='inner') \
                               .merge(Dim_Geography, on='Geography_ID', how='inner') \
                               .merge(Dim_TransactionType, on='TransactionType_ID', how='inner')

In [6]:
# ANALYTICAL QUESTIONS (POINT 2.2)

# Q1: What are the top 5 sectors by number of SELL transactions in US during 2024?

# Filter: Year 2024, SELL, United States of America
q1_filter = (df_analysis['Year'] == 2024) & \
            (df_analysis['TransactionType'] == 'SELL') & \
            (df_analysis['Country_Name'] == 'United States of America')

q1_result = df_analysis[q1_filter].groupby('Sector')['IDTransaction'].count().reset_index()
q1_result = q1_result.sort_values(by='IDTransaction', ascending=False).head(5)
q1_result.rename(columns={'IDTransaction': 'Total SELL Transactions'}, inplace=True)

print("1. Top 5 sectors by number of SELL transactions in US (2024):")
display(q1_result.style.hide(axis='index'))



1. Top 5 sectors by number of SELL transactions in US (2024):


Sector,Total SELL Transactions
Technology,158
Communication Services,58
Financial Services,55
Healthcare,50
Consumer Cyclical,48


In [7]:
# Q2: What are the top 5 industries by number of BUY transactions in Q4 of 2024?

# Filter: Year 2024, Quarter 4, BUY
q2_filter = (df_analysis['Year'] == 2024) & \
            (df_analysis['Quarter'] == 4) & \
            (df_analysis['TransactionType'] == 'BUY')

q2_result = df_analysis[q2_filter].groupby('Industry')['IDTransaction'].count().reset_index()
q2_result = q2_result.sort_values(by='IDTransaction', ascending=False).head(5)
q2_result.rename(columns={'IDTransaction': 'Total_BUY_Transactions'}, inplace=True)

print("2. Top 5 industries by number of BUY transactions in Q4 of 2024:")
display(q2_result.style.hide(axis='index'))


2. Top 5 industries by number of BUY transactions in Q4 of 2024:


Industry,Total_BUY_Transactions
Semiconductors,18
Internet Content & Information,15
Software - Infrastructure,10
Internet Retail,8
Diagnostics & Research,7


In [8]:
# Q3: Rank all quarters of 2024 by total number of transactions (BUY + SELL).

# Filter: Year 2024
q3_filter = (df_analysis['Year'] == 2024)

q3_result = df_analysis[q3_filter].groupby('Quarter')['IDTransaction'].count().reset_index()
q3_result = q3_result.sort_values(by='IDTransaction', ascending=False)
q3_result.rename(columns={'IDTransaction': 'Total_Transactions'}, inplace=True)

print("3. Rank all quarters of 2024 by total number of transactions:")
display(q3_result.style.hide(axis='index'))


3. Rank all quarters of 2024 by total number of transactions:


Quarter,Total_Transactions
1.000000,999
2.000000,542
3.000000,268
4.000000,260


In [9]:
# Q4: What are the top 10 countries by number of SELL transactions in 2024?

# Filter: Year 2024, SELL
q4_filter = (df_analysis['Year'] == 2024) & \
            (df_analysis['TransactionType'] == 'SELL')

q4_result = df_analysis[q4_filter].groupby('Country_Name')['IDTransaction'].count().reset_index()
q4_result = q4_result.sort_values(by='IDTransaction', ascending=False).head(10)
q4_result.rename(columns={'IDTransaction': 'Total_SELL_Transactions'}, inplace=True)

print("4. Top 10 countries by number of SELL transactions in 2024:")
display(q4_result.style.hide(axis='index'))


4. Top 10 countries by number of SELL transactions in 2024:


Country_Name,Total_SELL_Transactions
United States of America,389
United Kingdom of Great Britain and Northern Ireland,130
China,112
Brazil,69
"Taiwan, Province of China",50
"Netherlands, Kingdom of the",46
Switzerland,37
Ireland,31
Luxembourg,27
Canada,22


In [10]:
# Q5: What are the top 5 regions by total units bought in 2024?

# Filter: Year 2024, BUY
q5_filter = (df_analysis['Year'] == 2024) & \
            (df_analysis['TransactionType'] == 'BUY')

q5_result = df_analysis[q5_filter].groupby('Region')['Unit'].sum().reset_index()
q5_result = q5_result.sort_values(by='Unit', ascending=False).head(5)
q5_result.rename(columns={'Unit': 'Total_Units_Bought'}, inplace=True)

print("5. Top 5 regions by total units bought in 2024:")
display(q5_result.style.hide(axis='index'))

5. Top 5 regions by total units bought in 2024:


Region,Total_Units_Bought
Americas,37026.000000
Europe,22528.000000
Asia,9198.000000


In [11]:
# Export the dataframe in a CSV file for the Streamlit Dashboard
df_analysis.to_csv('dashboard_data.csv', index=False)